Phase 2

In [1]:
import pandas as pd
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

print("Machine Learning Libraries loaded!")

Machine Learning Libraries loaded!


In [2]:
# Loading the cleaned datasets
anime_df = pd.read_csv('data/cleaned_anime.csv')
print(f"Loaded {anime_df.shape[0]} anime titles.")

Loaded 12232 anime titles.


In [3]:
ps = PorterStemmer()

def stem_tags(text):
    # Split the genres by comma, stem them, and rejoin with spaces
    stemmed_words = []
    for word in str(text).split(','):
        stemmed_words.append(ps.stem(word.strip().lower()))
    return " ".join(stemmed_words)

# Creating a new column called 'tags' which our model will use
anime_df['tags'] = anime_df['genre'].apply(stem_tags)

# Show the before and after for the first 5 rows
anime_df[['name', 'genre', 'tags']].head()

,name,genre,tags
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural",drama romanc school supernatur
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",action adventur drama fantasi magic militari s...
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",action comedi histor parodi samurai sci-fi sho...
3,Steins;Gate,"Sci-Fi, Thriller",sci-fi thriller
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",action comedi histor parodi samurai sci-fi sho...


In [4]:
# 1. Initializing the Vectorizer (Removing common English stop-words)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# 2. Converting tags into numerical vectors
vector_matrix = tfidf.fit_transform(anime_df['tags']).toarray()

# 3. Calculating the Cosine Similarity Matrix
similarity_matrix = cosine_similarity(vector_matrix)

print(f"Similarity Matrix Shape: {similarity_matrix.shape}")
print("Math calculations completed successfully!")

Similarity Matrix Shape: (12232, 12232)
Math calculations completed successfully!


In [5]:
def test_recommendation(anime_title):
    # Find the row index of the anime
    anime_index = anime_df[anime_df['name'] == anime_title].index[0]
    
    # Get the similarity scores for this anime
    distances = similarity_matrix[anime_index]
    
    # Sort the scores from highest to lowest and grab the top 5
    anime_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Because you watched {anime_title}, we recommend:")
    print("-" * 30)
    for i in anime_list:
        # i[0] is the index, i[1] is the similarity score
        print(anime_df.iloc[i[0]]['name'])

# Testing
test_recommendation('One Piece')

Because you watched One Piece, we recommend:
------------------------------
One Piece: Episode of Merry - Mou Hitori no Nakama no Monogatari
One Piece: Episode of Nami - Koukaishi no Namida to Nakama no Kizuna
One Piece: Episode of Sabo - 3 Kyoudai no Kizuna Kiseki no Saikai to Uketsugareru Ishi
One Piece Film: Strong World Episode 0
One Piece: Episode of Luffy - Hand Island no Bouken


Phase 3

In [6]:
from surprise import Reader, Dataset, SVD
import pickle

print("Surprise library loaded!")

Surprise library loaded!


In [7]:
# Loading the cleaned rating data we saved in Phase 1
rating_df = pd.read_csv('data/cleaned_rating.csv')

# Defining the rating scale and loading the dataframe into Surprise
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(rating_df[['user_id', 'anime_id', 'rating']], reader)

# Converting it into a massive training matrix
trainset = data.build_full_trainset()

print("Data successfully formatted for SVD!")

Data successfully formatted for SVD!


In [8]:
# Initializing the SVD algorithm (n_factors is the number of hidden patterns it looks for)
svd_model = SVD(n_factors=100, random_state=42)

# Training the model
print("Training SVD Model...")
svd_model.fit(trainset)

print("SVD Model trained successfully!")

Training SVD Model...
SVD Model trained successfully!


In [9]:
# Save the Dataframe, the Similarity Matrix (Phase 2), and the SVD Model (Phase 3)
pickle.dump(anime_df, open('data/anime_data.pkl', 'wb'))
pickle.dump(similarity_matrix, open('data/similarity_matrix.pkl', 'wb'))
pickle.dump(svd_model, open('data/svd_model.pkl', 'wb'))

print("All models successfully saved as .pkl files! Phase 2 & 3 COMPLETE!")

All models successfully saved as .pkl files! Phase 2 & 3 COMPLETE!
